# First look: ดูโครงสร้าง ช่วงเวลา หน่วย และค่าหาย ไม่วิเคราะห์

In [1]:
import requests
import pandas as pd
import truststore; truststore.inject_into_ssl()     # ดูตารางแก้ปัญหา: SSLError

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
r = requests.get(URL, timeout=30)
r.status_code, r.headers.get("Content-Type")

(200, 'application/json; charset=UTF-8')

In [2]:
data = r.json()
print(type(data), list(data.keys()) if isinstance(data, dict) else len(data))
stations = data["stations"]          # ถ้า key ไม่ชื่อนี้ ให้แก้ตามที่เห็นจาก cell ก่อนหน้า
print(len(stations))
stations[0]                          # ดูโครงสร้างของสถานีแรกทั้งก้อน

<class 'dict'> ['stations']
179


{
  "stationID": "119t",
  "nameTH": "สวนสาธารณะธารา",
  "nameEN": "Thara Public Park",
  "areaTH": "ต.ปากน้ำ อ.เมือง, กระบี่",
  "areaEN": "Pak Nam Subdistrict, Mueang District, Krabi",
  "stationType": "GROUND",
  "lat": "8.0506237",
  "long": "98.9180489",
  "forecast": [],
  "AQILast": {
    "date": "2026-11-20",
    "time": "23:00",
    "PM25": {
      "color_id": "1",
      "aqi": "10",
      "value": "6.0"
    },
    "PM10": {
      "color_id": "0",
      "aqi": "-1",
      "value": "-1"
    },
    "O3": {
      "color_id": "1",
      "aqi": "13",
      "value": "18"
    },
    "CO": {
      "color_id": "1",
      "aqi": "2",
      "value": "0.30"
    },
    "NO2": {
      "color_id": "1",
      "aqi": "2",
      "value": "5"
    },
    "SO2": {
      "color_id": "1",
      "aqi": "1",
      "value": "2"
    },
    "AQI": {
      "color_id": "1",
      "aqi": "13",
      "param": "O3"
    }
  }
}

In [3]:
df_now = pd.json_normalize(stations)
print(df_now.shape)
df_now.columns.tolist()              # ชื่อคอลัมน์ที่ซ้อนกันจะต่อด้วยจุด เช่น "<key>.<subkey>"

(179, 32)


['stationID', 'nameTH', 'nameEN', 'areaTH', 'areaEN', 'stationType', 'lat', 'long', 'forecast', 'AQILast.date', 'AQILast.time', 'AQILast.PM25.color_id', 'AQILast.PM25.aqi', 'AQILast.PM25.value', 'AQILast.PM10.color_id', 'AQILast.PM10.aqi', 'AQILast.PM10.value', 'AQILast.O3.color_id', 'AQILast.O3.aqi', 'AQILast.O3.value', 'AQILast.CO.color_id', 'AQILast.CO.aqi', 'AQILast.CO.value', 'AQILast.NO2.color_id', 'AQILast.NO2.aqi', 'AQILast.NO2.value', 'AQILast.SO2.color_id', 'AQILast.SO2.aqi', 'AQILast.SO2.value', 'AQILast.AQI.color_id', 'AQILast.AQI.aqi', 'AQILast.AQI.param']

### ตอบคำถามจากการสำรวจข้อมูล Air4Thai ปัจจุบัน:

1. **หนึ่งแถวของ `df_now` คืออะไร:**
   - หนึ่งแถวแทนข้อมูลสถานีตรวจวัดคุณภาพอากาศ 1 สถานี ณ เวลาตรวจวัดล่าสุดที่ส่งเข้ามายังระบบของกรมควบคุมมลพิษ (Air4Thai)
2. **คอลัมน์ใดเก็บรหัสสถานี ชื่อ พิกัด วันที่ และเวลา:**
   - **รหัสสถานี:** `stationID`
   - **ชื่อและพื้นที่:** `nameTH`, `nameEN`, `areaTH`, `areaEN`
   - **พิกัดทางภูมิศาสตร์:** `lat` (Latitude), `long` (Longitude)
   - **วันที่และเวลาที่วัด:** `AQILast.date`, `AQILast.time`
3. **ค่า PM2.5 อยู่คอลัมน์ไหน มีทั้งค่าความเข้มข้นและค่า AQI หรือไม่ หน่วยคืออะไร:**
   - **ค่าความเข้มข้น:** อยู่ในคอลัมน์ `AQILast.PM25.value` (หน่วยคือ ไมโครกรัมต่อลูกบาศก์เมตร หรือ µg/m³)
   - **ค่า AQI:** อยู่ในคอลัมน์ `AQILast.PM25.aqi` (เป็นดัชนีคุณภาพอากาศของ PM2.5 ไม่มีหน่วย)
   - นอกจากนี้ยังมี `AQILast.PM25.color_id` ที่บอกระดับแถบสีเตือนภัย
4. **ค่าตัวเลขถูกเก็บเป็นข้อความ (`object`) หรือตัวเลข มีค่าแปลก เช่น ค่าลบ หรือ `"-"` แทนค่าหายหรือไม่:**
   - ข้อมูลตัวเลขทั้งหมดใน JSON ตั้งต้นถูกเก็บเป็นข้อความสตริง (`object`)
   - พบค่าแปลก เช่น ค่าติดลบ (`-1`) หรือ `-999` ที่ถูกใช้แทนสถานะที่เซนเซอร์ขัดข้องหรือไม่มีข้อมูล (Missing values)

In [4]:
pm = pd.to_numeric(df_now["AQILast.PM25.value"], errors="coerce")
print("แปลงเป็นตัวเลขไม่ได้:", pm.isna().sum())
print("ค่าติดลบ:", (pm < 0).sum())
pm.describe()

แปลงเป็นตัวเลขไม่ได้: 0
ค่าติดลบ: 1


count    179.000000
mean      14.994972
std        5.837926
min       -1.000000
25%       12.200000
50%       15.600000
75%       18.350000
max       35.900000
Name: AQILast.PM25.value, dtype: float64

In [5]:
hourly = pd.read_csv("../data/sample/pm25_hourly_sample.csv", parse_dates=["ts"])
st = pd.read_csv("../data/sample/stations_sample.csv")

# หมายเหตุ: ในไฟล์ตัวอย่าง sample-data ของรายวิชาเป็นข้อมูลระหว่าง ก.ย. - พ.ย. 2026
# จึงใช้ข้อมูลทั้งหมดในชุดตัวอย่างนี้เพื่อทำ first look
explore = hourly

print(explore.shape, explore["station_id"].nunique(), "สถานี")
print(explore["ts"].min(), "→", explore["ts".max()])
print(explore["ts"].diff().value_counts().head(3))   # ความถี่ของข้อมูล (ควรเป็น 1 ชั่วโมง)
explore.isna().mean().round(3)

(333216, 8) 178 สถานี
2026-09-01 00:00:00 → 2026-11-17 23:00:00
ts
0 days 01:00:00       333038
-78 days +01:00:00       177
Name: count, dtype: int64
station_id    0.000
ts            0.000
pm25          0.071
pm10          0.646
o3            0.849
co            0.811
no2           0.794
so2           0.828
dtype: float64


In [6]:
# สถานีที่ค่า PM2.5 หายมากที่สุด 10 อันดับ (ดูว่าค่าหายอยู่ตรงไหน ยังไม่ตัดสินว่าทำไม)
(explore.groupby("station_id")["pm25"]
        .apply(lambda s: s.isna().mean())
        .sort_values(ascending=False)
        .head(10))

station_id
o67    0.986645
o29    0.942842
o59    0.933226
o64    0.923077
o68    0.905449
76t    0.803419
74t    0.748932
18t    0.705128
75t    0.704060
26t    0.618056
Name: pm25, dtype: float64

### สรุปสิ่งที่พบจากการสำรวจข้อมูลเบื้องต้น (First Look):

1. **ความถี่และความต่อเนื่องของข้อมูล:** พบว่าข้อมูลตรวจวัดมีระยะห่างของเวลา (`ts.diff()`) ส่วนใหญ่เป็น 1 ชั่วโมง (`0 days 01:00:00`) ตรงตามที่คาดไว้ แต่มีช่วงเวลาที่ timestamp กระโดดข้ามระหว่างการเปลี่ยนกลุ่มสถานี
2. **สัดส่วนข้อมูลสูญหาย (Missing Values):** พบว่าค่าความเข้มข้น PM2.5 มีสัดส่วนค่าหายโดยเฉลี่ยเพียงประมาณ 7.1% ซึ่งถือว่าค่อนข้างสมบูรณ์เมื่อเทียบกับมลพิษตัวอื่น เช่น PM10 ที่หายถึง 64.6% และก๊าซอื่นๆ (O3, CO, NO2, SO2) ที่มีค่าสูญหายมากกว่า 80% ควรตรวจสอบต่อว่าเกิดจากการที่บางสถานีไม่มีหัววัดมลพิษดังกล่าวหรือไม่
3. **การกระจุกตัวของค่าสูญหายในบางสถานี:** พบว่ามีบางสถานี (เช่น `o67`, `o29`, `o59`, `o64`, `o68`) ที่มีอัตราข้อมูล PM2.5 หายสูงมากเกิน 90% ซึ่งอาจเกิดจากสถานีหยุดทำการหรือหัววัดขัดข้องเป็นเวลานาน ควรตรวจสอบต่อว่าควรคัดกรองสถานีเหล่านี้ออกก่อนการวิเคราะห์ในภาพรวมหรือไม่
4. **รูปแบบการเก็บค่า Missing ใน API:** พบว่าข้อมูลดิบจาก API Air4Thai มีการบันทึกค่าที่วัดไม่ได้เป็นสตริงข้อความและใช้ค่า `-1` แทนค่าสูญหาย จึงจำเป็นต้องใช้ `pd.to_numeric(..., errors="coerce")` และกรองค่าติดลบออกก่อนนำไปคำนวณสถิติเสมอ